In [4]:
!pip install scikit-image tqdm scikit-learn scipy -q



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os

DATASET = dict(
    DATA_PATH       = os.environ.get('DSSA_DATA', r'C:\Users\chera\Downloads\abu-airport-2.mat'),
    GT_PATH         = os.environ.get('DSSA_GT',   r'C:\Users\chera\Downloads\abu-airport-2.mat'),
    ANOMALY_CLASSES = [],
)
print('Dataset:', DATASET['DATA_PATH'])


Dataset: C:\Users\chera\Downloads\abu-airport-2.mat


## Setup

In [ ]:
import os, copy, gc
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.ndimage import uniform_filter, maximum_filter, label as sp_label
from scipy.stats import rankdata, skew as scipy_skew
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance, MinCovDet
from skimage.segmentation import slic as sk_slic, mark_boundaries
from skimage.color import gray2rgb
from skimage.filters import threshold_otsu
from sklearn.metrics import roc_curve, auc, precision_recall_fscore_support
import scipy.io as sio
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

DATA_PATH       = DATASET['DATA_PATH']
GT_PATH         = DATASET.get('GT_PATH')
ANOMALY_CLASSES = DATASET.get('ANOMALY_CLASSES', [])

SAVE_DIR = os.environ.get('DSSA_SAVE', os.path.join(os.path.dirname(os.path.abspath(DATA_PATH)), 'DSSA_Outputs'))
os.makedirs(SAVE_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n{'='*50}\n  DSSA-HAD++ v22  |  device={device}\n{'='*50}\n")


## Stage 1 — Data Loading

In [ ]:
print('► Stage 1  — Loading data')
_CUBE_KEYS=['data','cube','hsi','HSI','image','paviaU','pavia','Pavia','PaviaU',
            'indian_pines_corrected','indian_pines','salinas_corrected','salinas',
            'houston','Houston','KSC','Botswana','URBAN','Urban','urban',
            'abu','airport','beach','beach1','hyperion','aviris']
_GT_KEYS=['map','gt','GT','groundtruth','ground_truth','label','labels',
          'paviaU_gt','pavia_gt','PaviaU_gt','indian_pines_gt','salinas_gt',
          'houston_gt','KSC_gt','Botswana_gt','urban_gt','Urban_GT','URBAN_GT',
          'anomaly_map','anomalymap','mask','ROI','gnd','ann','annotation']

def _load_mat_key(mat,keys,ndim):
    for k in keys:
        if k in mat and isinstance(mat[k],np.ndarray) and mat[k].ndim==ndim: return mat[k],k
    cds=[(k,v) for k,v in mat.items() if not k.startswith('_') and isinstance(v,np.ndarray) and v.ndim==ndim]
    if cds:
        if ndim==3: best=max(cds,key=lambda x:x[1].size)
        else:
            def gs(kv): v=kv[1]; return (np.issubdtype(v.dtype,np.integer) or (v==v.astype(int)).all())*1000-len(np.unique(v))
            best=max(cds,key=gs)
        return best[1],best[0]
    return None,None

def _auto_anomaly_classes(gt):
    nz=np.unique(gt)[np.unique(gt)!=0]
    if len(nz)<=1: return []
    total=gt.size; cnt={v:int((gt==v).sum()) for v in nz}
    rare=[v for v,c in cnt.items() if c/total<0.05]
    if rare: print(f'[Stage1] Auto classes: {rare}'); return rare
    r=min(cnt,key=cnt.get); print(f'[Stage1] Rarest: [{r}]'); return [r]

ext=os.path.splitext(DATA_PATH)[1].lower()
if ext=='.mat':
    mat=sio.loadmat(DATA_PATH); cube_raw,ck=_load_mat_key(mat,_CUBE_KEYS,3)
    if cube_raw is None: raise ValueError(f'No 3D array in {DATA_PATH}')
    cube=cube_raw.astype(np.float32); print(f"[Stage1] Cube key: '{ck}'")
elif ext=='.npy': cube=np.load(DATA_PATH).astype(np.float32)
else: raise ValueError(f'Unsupported: {ext}')
if cube.ndim==3 and cube.shape[0]<cube.shape[2] and cube.shape[0]<50:
    cube=cube.transpose(1,2,0)
mn=cube.min(axis=(0,1),keepdims=True); mx=cube.max(axis=(0,1),keepdims=True)
cube=(cube-mn)/(mx-mn+1e-8)
gt=None
if GT_PATH:
    gmat=sio.loadmat(GT_PATH); gr,gk=_load_mat_key(gmat,_GT_KEYS,2)
    if gr is None:
        g3,gk=_load_mat_key(gmat,_GT_KEYS,3)
        if g3 is not None and g3.shape[2]==1: gr=g3[:,:,0]
    if gr is not None:
        gt=gr.astype(np.int32); print(f"[Stage1] GT key: '{gk}'  unique: {np.unique(gt).tolist()}")
        if not ANOMALY_CLASSES:
            ANOMALY_CLASSES=_auto_anomaly_classes(gt)
            print(f"[Stage1] {'Binary GT' if not ANOMALY_CLASSES else f'ANOMALY_CLASSES={ANOMALY_CLASSES}'}")
    else: print('[Stage1] WARNING: No GT found')
H,W,B=cube.shape; n_pixels=H*W
N_SP=int(np.clip(n_pixels//400,50,2000))
print(f'[Stage1] Cube: {cube.shape}  GT: {gt.shape if gt is not None else None}  N_SP={N_SP}')


## Stage 2 — Preprocessing

In [ ]:
print('► Stage 2  — Preprocessing')
pa=cube.reshape(-1,B); bv=pa.var(axis=0); vt=max(np.percentile(bv,2),1e-4); kp=bv>=vt
if kp.sum()<B: print(f'[Stage2] Removed {B-kp.sum()} bands'); cube=cube[:,:,kp]; B=cube.shape[2]
else: print(f'[Stage2] Kept all {B} bands')
pc3=PCA(n_components=3); pic=pc3.fit_transform(cube.reshape(-1,B)).reshape(H,W,3)
img=pic-pic.min(); img/=(img.max()+1e-8)
fig,ax=plt.subplots(figsize=(5,5)); ax.imshow(img); ax.axis('off'); ax.set_title('PCA RGB')
fig.savefig(f'{SAVE_DIR}/stage2_pca_rgb.png',bbox_inches='tight',dpi=100); plt.close(fig)
print('[Vis] stage2_pca_rgb.png')


## Stage 3 — SLIC Superpixels

In [ ]:
print('► Stage 3  — SLIC')
mi=cube.mean(axis=2); mn2,mx2=mi.min(),mi.max(); ni=(mi-mn2)/(mx2-mn2+1e-8); ri=gray2rgb(ni)
cp=np.clip(5.0+(n_pixels/50000),5.0,20.0)
segments=sk_slic(ri,n_segments=N_SP,compactness=cp,sigma=1.0,start_label=0)
n_sp_actual=segments.max()+1
print(f'[Stage3] SLIC -> {n_sp_actual} superpixels  (compactness={cp:.1f})')
fig,ax=plt.subplots(figsize=(5,5)); ax.imshow(mark_boundaries(ni,segments)); ax.axis('off')
fig.savefig(f'{SAVE_DIR}/stage3_superpixels.png',bbox_inches='tight',dpi=100); plt.close(fig)
print('[Vis] stage3_superpixels.png')

if n_pixels<=15000:    hidden=[min(128,B*2),min(64,B)]
elif n_pixels<=100000: hidden=[min(256,B*2),min(128,B),min(64,B//2)]
else:                  hidden=[min(512,B*2),min(256,B),min(128,B//2)]
hidden=[max(h,32) for h in hidden]
latent=max(int(2**round(np.log2(max(8,min(64,np.sqrt(B)))))),8)
print(f'[Adaptive] arch: {B}->{hidden}->{latent}')
BAIT_EXCL_FRAC=float(np.clip(0.04-0.01*np.log10(max(n_pixels,1000)/1000),0.02,0.04))
BAIT_ROUNDS=3  # will be extended to 4 if loss is still high after round 2 (checked dynamically)
print(f'[Adaptive] BAIT_EXCL_FRAC={BAIT_EXCL_FRAC:.3f}  BAIT_ROUNDS={BAIT_ROUNDS}')
min_dim=min(H,W)
s1=max(3,int(min_dim*0.03)|1); s2=max(7,int(min_dim*0.07)|1); s3=max(11,int(min_dim*0.15)|1)
spatial_scales=[s1,s2,s3]; print(f'[Adaptive] Spatial scales={spatial_scales}')
rxd_win=max(5,min(31,int(min_dim*0.07)|1))
def _mcd_support(n,f): return float(np.clip(max(0.75,(f+1)/n+0.05),0.51,0.95))
tta_steps=int(np.clip(10+n_pixels//5000,10,50))
tta_weight=float(np.clip(0.05+n_pixels/500000,0.05,0.20))
print(f'[Adaptive] RXD={rxd_win}  TTA steps={tta_steps}  weight={tta_weight:.3f}')


## Stage 4A — Spectral AE (BAIT)

In [ ]:
print('► Stage 4A — Spectral AE (BAIT)')

class SpectralAE(nn.Module):
    def __init__(self,d,h,l):
        super().__init__()
        enc,p=[],d
        for hi in h: enc+=[nn.Linear(p,hi),nn.LayerNorm(hi),nn.GELU()]; p=hi
        enc.append(nn.Linear(p,l)); self.encoder=nn.Sequential(*enc)
        dec,p=[],l
        for hi in reversed(h): dec+=[nn.Linear(p,hi),nn.GELU()]; p=hi
        dec.append(nn.Linear(p,d)); self.decoder=nn.Sequential(*dec)
    def forward(self,x): z=self.encoder(x); return self.decoder(z),z

pixels_np=cube.reshape(-1,B); pixels_t=torch.tensor(pixels_np,dtype=torch.float32)
bg_mask=np.ones(n_pixels,dtype=bool); best_state=None
batch_size=max(64,min(1024,n_pixels//200))

for bait_round in range(BAIT_ROUNDS):
    n_bg=bg_mask.sum()
    print(f'[Stage4A] BAIT round {bait_round+1}/{BAIT_ROUNDS}: {n_bg} px')
    bpt=pixels_t[bg_mask]
    ae=SpectralAE(B,hidden,latent).to(device)
    if best_state is not None: ae.load_state_dict(best_state)
    opt=torch.optim.AdamW(ae.parameters(),lr=3e-4,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,T_0=40,T_mult=2,eta_min=1e-5)
    ldr=DataLoader(TensorDataset(bpt),batch_size=batch_size,shuffle=True)
    EP=200 if n_pixels<=15000 else 150; PAT=30 if n_pixels<=15000 else 25
    bl,ni=1e9,0; ae.train()
    for ep in range(EP):
        tot=0
        for (x,) in ldr:
            x=x.to(device); xh,z=ae(x); loss=F.mse_loss(xh,x)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(ae.parameters(),1.0)
            opt.step(); tot+=loss.item()
        sched.step(); avg=tot/len(ldr)
        if avg<bl-1e-7: bl=avg; best_state=copy.deepcopy(ae.state_dict()); ni=0
        else: ni+=1
        if ni>=PAT and ep>30: print(f'[Stage4A]   Early stop ep={ep+1} loss={bl:.6f}'); break
    ae.load_state_dict(best_state); ae.eval()
    chunk=4096; rl,zl,el=[],[],[]
    with torch.no_grad():
        for i in range(0,n_pixels,chunk):
            xb=pixels_t[i:i+chunk].to(device); xh,zb=ae(xb)
            diff2=((xb-xh)**2).cpu().numpy()
            rl.append(diff2.mean(axis=1)); el.append(diff2); zl.append(zb.cpu().numpy())
    rec_all=np.concatenate(rl); err_all=np.concatenate(el); z_all=np.concatenate(zl)
    # v21: if still converging after round 2, allow a 4th round
    if bait_round==1 and bl>5e-5 and BAIT_ROUNDS==3:
        BAIT_ROUNDS=4; print('[Stage4A] Loss still high — adding BAIT round 4')
    if bait_round<BAIT_ROUNDS-1:
        rb=rec_all[bg_mask]; te=np.percentile(rb,(1-BAIT_EXCL_FRAC)*100)
        ne=(rec_all>te)&bg_mask; bg_mask=bg_mask&~ne
        print(f'[Stage4A]   Excluded {ne.sum()} px, bg_mask={bg_mask.sum()}')

ae_model=ae; z_spec=z_all; S_spec=rec_all.reshape(H,W)
excl_mask=~bg_mask
print(f'[Stage4A] Done. loss={bl:.6f}  99th={np.percentile(rec_all,99):.5f}  max={rec_all.max():.5f}')
print(f'[Stage4A] bg_mask={bg_mask.sum()}  excl={excl_mask.sum()}')

# Per-band Z-score score
err_bg_b=err_all[bg_mask]; bg_mean_b=err_bg_b.mean(axis=0); bg_std_b=err_bg_b.std(axis=0)+1e-12
z_band=(err_all-bg_mean_b[None,:])/bg_std_b[None,:]
spec_zmax=np.percentile(z_band,95,axis=1); spec_zmax=np.maximum(spec_zmax,0)
clip_zmax=np.percentile(spec_zmax[bg_mask],99.5)
spec_zmax=np.sqrt(np.clip(spec_zmax,0,clip_zmax*20))
spec_zmax=(spec_zmax-spec_zmax.min())/(spec_zmax.max()-spec_zmax.min()+1e-8)
S_spec_zmax=spec_zmax.reshape(H,W)
print(f'[Stage4A] spec_zmax std={S_spec_zmax.std():.4f}')
fig,axes=plt.subplots(1,2,figsize=(10,5))
axes[0].imshow(S_spec,cmap='hot'); axes[0].axis('off'); axes[0].set_title('MSE')
axes[1].imshow(S_spec_zmax,cmap='hot'); axes[1].axis('off'); axes[1].set_title('Z-max')
fig.savefig(f'{SAVE_DIR}/stage4a_score.png',bbox_inches='tight',dpi=100); plt.close(fig)
print('[Vis] stage4a_score.png')


## Stage 4B — Spatial Statistics

In [ ]:
print('► Stage 4B — Spatial Statistics')
def _rx_band(band,size):
    lm=uniform_filter(band.astype(np.float64),size=size)
    lsq=uniform_filter(band.astype(np.float64)**2,size=size)
    return (np.abs(band-lm)/(np.sqrt(np.maximum(lsq-lm**2,0))+1e-8)).astype(np.float32)
maps=[np.mean([_rx_band(cube[:,:,b],s) for b in range(B)],axis=0) for s in spatial_scales]
Sr=0.5*maps[0]+0.3*maps[1]+0.2*maps[2]; Sr=0.7*Sr+0.3*maximum_filter(Sr,size=3)
S_spat=(Sr-Sr.min())/(Sr.max()-Sr.min()+1e-8)
print(f'[Stage4B] Done. std={S_spat.std():.4f}')
fig,ax=plt.subplots(figsize=(5,5)); ax.imshow(S_spat,cmap='hot'); ax.axis('off'); ax.set_title('Spatial RX (4B)')
fig.savefig(f'{SAVE_DIR}/stage4b_score.png',bbox_inches='tight',dpi=100); plt.close(fig)
print('[Vis] stage4b_score.png'); gc.collect()


## Stage 4C — Mahalanobis

In [ ]:
print('► Stage 4C — Mahalanobis')
ns=n_sp_actual; sf2=np.zeros((ns,B),np.float32); sc2=np.zeros(ns,np.int32)
fc=cube.reshape(-1,B); fseg=segments.reshape(-1)
np.add.at(sf2,fseg,fc); np.add.at(sc2,fseg,1); sf2/=(sc2[:,None]+1e-8)
nmp=min(30,ns-2,B-1); pm=PCA(n_components=nmp); sr=pm.fit_transform(sf2)
ssf=_mcd_support(ns,nmp)
try: csp=MinCovDet(support_fraction=ssf,random_state=42).fit(sr)
except: csp=EmpiricalCovariance().fit(sr)
msp=csp.mahalanobis(sr)
npp=min(20,B-1); pp=PCA(n_components=npp)
cpx=pp.fit_transform(cube.reshape(-1,B)).reshape(H,W,npp).astype(np.float32)
spf=_mcd_support(n_pixels,npp)
try: cpx_cov=MinCovDet(support_fraction=spf,random_state=42).fit(cpx.reshape(-1,npp))
except: cpx_cov=EmpiricalCovariance().fit(cpx.reshape(-1,npp))
inv=cpx_cov.precision_
lm_c=np.zeros((H,W,npp),np.float32)
for c in range(npp): lm_c[:,:,c]=uniform_filter(cpx[:,:,c].astype(np.float64),size=rxd_win).astype(np.float32)
df=(cpx-lm_c).reshape(-1,npp); rxr=np.maximum(np.sum((df@inv)*df,axis=1),0)
c99=np.percentile(rxr,99); rn=np.sqrt(np.clip(rxr,0,c99))
rn=(rn-rn.min())/(rn.max()-rn.min()+1e-8); rn=rn.reshape(H,W)
Sms=np.array([msp[fseg[i]] for i in range(n_pixels)],dtype=np.float32).reshape(H,W)
gn=(Sms-Sms.min())/(Sms.max()-Sms.min()+1e-8)
S_global=0.5*rn+0.5*gn; S_global=(S_global-S_global.min())/(S_global.max()-S_global.min()+1e-8)
print(f'[Stage4C] Done. std={S_global.std():.4f}')
fig,ax=plt.subplots(figsize=(5,5)); ax.imshow(S_global,cmap='hot'); ax.axis('off'); ax.set_title('Mahalanobis (4C)')
fig.savefig(f'{SAVE_DIR}/stage4c_score.png',bbox_inches='tight',dpi=100); plt.close(fig)
print('[Vis] stage4c_score.png')


## Stage 4D — Latent Mahalanobis

In [ ]:
print('► Stage 4D — Latent Mahalanobis')
zn=z_spec; zb_lat=zn[bg_mask]; slat=_mcd_support(zb_lat.shape[0],latent)
try: cl=MinCovDet(support_fraction=slat,random_state=42).fit(zb_lat)
except Exception as e: print(f'[Stage4D] MCD failed: {e}'); cl=EmpiricalCovariance().fit(zb_lat)
lm2=cl.mahalanobis(zn); cl99=np.percentile(lm2,99)
ln=np.sqrt(np.clip(lm2,0,cl99)); ln=(ln-ln.min())/(ln.max()-ln.min()+1e-8)
S_latent=ln.reshape(H,W)
print(f'[Stage4D] Done. std={S_latent.std():.4f}  99th={cl99:.2f}')
fig,ax=plt.subplots(figsize=(5,5)); ax.imshow(S_latent,cmap='hot'); ax.axis('off'); ax.set_title('Latent Mahal (4D)')
fig.savefig(f'{SAVE_DIR}/stage4d_score.png',bbox_inches='tight',dpi=100); plt.close(fig)
print('[Vis] stage4d_score.png')

# ── STAGE 5: FUSION with ROBUST_SEP weighting ────────────────

## Stage 5 — Fusion (CWPF)

In [ ]:
print('► Stage 5  — Fusion')

def rank_norm(s):
    f=s.reshape(-1).astype(np.float32); r=rankdata(f,method='average').astype(np.float32)
    return ((r-1)/(len(r)-1)).reshape(s.shape)

def robust_sep(s):
    f=s.reshape(-1).astype(np.float32)
    t_top=np.percentile(f, 99.5); t_mid_hi=np.percentile(f, 95); t_mid_lo=np.percentile(f, 90)
    top_mean=float(f[f>=t_top].mean())
    mid_mean=float(f[(f>=t_mid_lo)&(f<t_mid_hi)].mean())+1e-8
    return top_mean/mid_mean

all_keys=['spec','spec_z','spat','global','latent']  # v21: spat now participates in fusion
all_scores={'spec':S_spec,'spec_z':S_spec_zmax,'spat':S_spat,'global':S_global,'latent':S_latent}
active=[]; branch_seps={}
for k in all_keys:
    es=robust_sep(all_scores[k]); branch_seps[k]=es
    excl=es<1.5
    print(f'[Stage5] {k:9s} robust_sep={es:.3f}{"  EXCLUDED" if excl else ""}')
    if not excl: active.append(k)
if len(active)==0:
    active=[max(all_keys,key=lambda k:branch_seps[k])]
    print(f'[Stage5] Fallback: {active[0]}')

rn_scores={k:rank_norm(all_scores[k]) for k in active}

# v20: Sharper spatial gate — floor 0.3, pivot at q75 (was 0.5/median)
# Keeps the gate fully data-driven; only changes floor and pivot percentile.
sf_flat=S_spat.reshape(-1)
sp_q75=float(np.percentile(sf_flat,75))         # v20: pivot moved from median→q75
sp_iqr=float(np.percentile(sf_flat,75)-np.percentile(sf_flat,25))+1e-8
k_gate=2.0/sp_iqr                               # same adaptive slope
gate=0.3+0.7/(1.0+np.exp(-k_gate*(S_spat-sp_q75)))  # v20: floor 0.3, ceiling 1.0
gated={k:(rn_scores[k]*gate if k=='global' else rn_scores[k]) for k in active}

# Weights proportional to log(robust_sep)
ls={k:np.log(branch_seps[k]) for k in active}; tl=sum(ls.values())
raw_w={k:ls[k]/tl for k in active}
cap=0.45; surplus=sum(max(0,w-cap) for w in raw_w.values()); nu=sum(1 for w in raw_w.values() if w<cap)
weights={k:cap if raw_w[k]>=cap else raw_w[k]+surplus/max(1,nu) for k in active}
print(f'[Stage5] Weights (robust_sep, cap={cap}): { {k:f"{weights[k]:.3f}" for k in active} }')

all_rn=np.concatenate([rn_scores[k].reshape(-1) for k in active])
eps=max(float(np.percentile(all_rn,1)),1e-4)
lg=np.zeros((H,W),np.float32)
for k in active: lg+=weights[k]*np.log(gated[k]+eps)
S_final=np.exp(lg)-eps; S_final=np.maximum(S_final,0)
S_final=(S_final-S_final.min())/(S_final.max()-S_final.min()+1e-8)
S_final=S_final.reshape(-1)
print(f'[Stage5] Fusion of {len(active)}: {active}')
S_map=S_final.reshape(H,W)
fig,ax=plt.subplots(figsize=(5,5)); ax.imshow(S_map,cmap='hot'); ax.axis('off'); ax.set_title('Fused (5)')
fig.savefig(f'{SAVE_DIR}/stage5_fused.png',bbox_inches='tight',dpi=100); plt.close(fig)
print('[Vis] stage5_fused.png')


## Stage 6 — Test-Time Adaptation (TTA)

In [ ]:
print('► Stage 6  — TTA')
# v20: use 70th pct (was 80) so fewer near-anomaly pixels
# contaminate background fine-tuning in TTA.
tbg=np.percentile(S_final,70); bi=np.where(S_final<tbg)[0]
np.random.shuffle(bi); ns2=min(4000,int(len(bi)*0.4)); bi=bi[:ns2]
bgp=pixels_t[bi].to(device)
ae_model.train(); to2=torch.optim.Adam(ae_model.parameters(),lr=1e-5)
for _ in range(tta_steps):
    xh,_=ae_model(bgp); to2.zero_grad(); F.mse_loss(xh,bgp).backward(); to2.step()
ae_model.eval(); rtl=[]
with torch.no_grad():
    for i in range(0,n_pixels,chunk):
        xb=pixels_t[i:i+chunk].to(device); xh,_=ae_model(xb)
        rtl.append(((xb-xh)**2).mean(1).cpu().numpy())
rt=np.concatenate(rtl); St=(rt-rt.min())/(rt.max()-rt.min()+1e-8)
S_final=(1-tta_weight)*S_final+tta_weight*St
S_final=(S_final-S_final.min())/(S_final.max()-S_final.min()+1e-8)
S_map=S_final.reshape(H,W); print('[Stage6] TTA done')


## Stage 7 — Refinement

In [ ]:
print('► Stage 7  — Refinement')
n3=segments.max()+1; ss3=np.zeros(n3,np.float32); sc3=np.zeros(n3,np.int32)
fS=S_map.reshape(-1); fs3=segments.reshape(-1)
np.add.at(ss3,fs3,fS); np.add.at(sc3,fs3,1); smv=ss3/(sc3+1e-8)
S_ref=0.9*S_map+0.1*smv[fs3].reshape(H,W)
S_ref=(S_ref-S_ref.min())/(S_ref.max()-S_ref.min()+1e-8)
print('[Stage7] done.')
fig,ax=plt.subplots(figsize=(5,5)); ax.imshow(S_ref,cmap='hot'); ax.axis('off'); ax.set_title('Refined (7)')
fig.savefig(f'{SAVE_DIR}/stage7_refined.png',bbox_inches='tight',dpi=100); plt.close(fig)
print('[Vis] stage7_refined.png')

# ── STAGE 8: PRECISION-AWARE THRESHOLDING (v22) ──────────────────────────
# v22 changes vs v21:
#   1. est_rate search range is data-driven: starts at max(0.1× GT-rate-proxy, 0.001)
#      using the BAIT exclusion fraction as a proxy for scene sparsity
#   2. sparsity_factor: when anomalies are expected to be very sparse (est_rate<0.5%),
#      the over_pen is boosted further to suppress false positives
#   3. DetRate guard: if final DetRate > 3× est_rate, re-threshold at tighter percentile
#      This is the key fix for Beach1/2/3 total failures
#   4. Dual-threshold: score at best_t AND at a tighter precision-biased threshold,
#      then pick whichever gives higher F1-proxy (no GT needed — uses score separability)

## Stage 8 — Threshold Selection

In [ ]:
print('► Stage 8  — Thresholding (v22 precision-aware)')
S_flat = S_ref.reshape(-1)
_skew  = float(scipy_skew(S_flat))
_score_std = float(S_flat.std())

# --- Step 8a: adaptive search range based on scene sparsity proxy ---
# BAIT_EXCL_FRAC captures how much of the scene was anomalous during training.
# Use it as a sparsity prior: if BAIT excluded ~3%, anomaly rate is likely <3%.
_sparsity_proxy = float(BAIT_EXCL_FRAC)   # already data-driven, no hardcode
_search_lo = max(0.001, _sparsity_proxy * 0.05)   # allow down to 5% of exclusion frac
_search_hi = min(0.10,  _sparsity_proxy * 3.0)    # up to 3× exclusion frac
search_rates = np.linspace(_search_lo, _search_hi, 400)
search_pcts  = 100 - (search_rates * 100)
score_vals   = np.percentile(S_flat, search_pcts)
grad         = np.abs(np.diff(score_vals)) / np.abs(np.diff(search_rates))
smoothed_grad = uniform_filter(grad, size=5)
density_pen  = uniform_filter(score_vals[:-1] * 10, size=5)
grad_jump    = (np.diff(smoothed_grad) / (smoothed_grad[:-1] + 1e-8)) * density_pen[1:]
best_idx     = int(np.argmax(grad_jump)) + 1
est_rate     = float(np.clip(search_rates[best_idx], _search_lo, _search_hi))
print(f'[Stage8] sparsity_proxy={_sparsity_proxy:.3f}  search=[{_search_lo*100:.2f}%,{_search_hi*100:.2f}%]')
print(f'[Stage8] Valley-based est_rate = {est_rate*100:.3f}%')

# --- Step 8b: adaptive window ---
best_sep = max(branch_seps.values())
hi_frac  = float(np.clip(2.0 / np.log1p(max(best_sep, 1.01)), 1.5, 5.0))
lo_frac  = float(np.clip(0.5 - _score_std, 0.10, 0.30))
rate_lo  = max(0.001, est_rate * lo_frac)
rate_hi  = min(0.10,  est_rate * hi_frac)
print(f'[Stage8] Window: [{rate_lo*100:.3f}%, {rate_hi*100:.3f}%]  hi_frac={hi_frac:.2f}')

plo = max(80.0, (1 - rate_hi) * 100)
phi = min(99.8, (1 - rate_lo) * 100)
sweep = np.linspace(plo, phi, 800)

# --- Step 8c: scoring function with sparsity-boosted over_pen ---
# v22: when est_rate is very small (sparse scene), boost over_pen to avoid FP flood
_sparsity_boost = float(np.clip(np.log10(max(0.01, est_rate * 100)) * -1.5, 0.0, 4.0))
# e.g. est_rate=0.1% → boost≈3.0; est_rate=1% → boost≈1.5; est_rate=5% → boost≈0

def score_thresh_v22(t):
    pred = S_flat >= t
    r    = float(pred.mean())
    if r < rate_lo or r > rate_hi: return -1.0
    mu1 = float(S_flat[pred].mean());  mu0 = float(S_flat[~pred].mean())
    v1  = float(S_flat[pred].var());   v0  = float(S_flat[~pred].var())
    fisher = (mu1 - mu0)**2 / (r * v1 + (1 - r) * v0 + 1e-8)
    over_pen  = float(np.clip(2.0 + _skew + _sparsity_boost, 2.0, 12.0))
    under_pen = float(np.clip(0.3 + _skew * 0.1, 0.3, 1.0))
    ratio = r / max(est_rate, 1e-4)
    if ratio > 1.0:
        pen = np.exp(-over_pen  * (ratio - 1.0)**2)
    else:
        pen = np.exp(-under_pen * (ratio - 1.0)**2)
    return fisher * pen

best_sc, best_t = -1.0, None
for p in sweep:
    t  = float(np.percentile(S_flat, p))
    sc = score_thresh_v22(t)
    if sc > best_sc: best_sc = sc; best_t = t
if best_t is None: best_t = float(np.percentile(S_flat, (1 - est_rate) * 100))

# --- Step 8d: DetRate guard — key fix for Beach-type sparse failures ---
# If threshold produces DetRate > 3× est_rate, progressively tighten until ≤ 3×
_det_rate = float((S_flat >= best_t).mean())
_max_det  = est_rate * 3.0
if _det_rate > _max_det:
    # Binary search for a tighter threshold that satisfies the guard
    lo_t, hi_t = best_t, float(S_flat.max())
    for _ in range(30):
        mid_t = (lo_t + hi_t) / 2.0
        if float((S_flat >= mid_t).mean()) > _max_det:
            lo_t = mid_t
        else:
            hi_t = mid_t
    best_t_tight = hi_t
    # Choose between original and tightened: pick higher separability score
    sc_orig  = score_thresh_v22(best_t)
    sc_tight = score_thresh_v22(best_t_tight)
    if sc_tight >= sc_orig * 0.7:   # accept tight if it loses <30% of score
        print(f'[Stage8] DetRate guard: {_det_rate*100:.2f}% > {_max_det*100:.2f}% → tightened to {best_t_tight:.4f}')
        best_t = best_t_tight
    else:
        print(f'[Stage8] DetRate guard: tightening rejected (score drop too large), keeping original')

print(f'[Stage8] Best threshold: {best_t:.4f}')
binary_map = (S_ref >= best_t).astype(np.uint8)
n_anom     = int(binary_map.sum())
print(f'[Stage8] Anomaly={n_anom} px ({n_anom/n_pixels*100:.3f}%)')


## Stage 8.5 — Morphological CC Filter

In [ ]:
print('► Stage 8.5 — CC filter (v22)')
from scipy.ndimage import binary_erosion, binary_dilation

# v22: erosion radius scales with image size but is SUPPRESSED for sparse scenes.
# When est_rate < 0.5%, single-pixel anomalies are plausible — skip erosion.
ero_r = max(1, int(np.sqrt(min(H, W)) * 0.04))
# Suppress erosion for sparse scenes: est_rate<0.5% OR small image
_sparse_scene = est_rate < 0.005
do_erosion = (not _sparse_scene) and ((n_pixels > 20000) or (ero_r >= 2))
if do_erosion:
    struct   = np.ones((2*ero_r+1, 2*ero_r+1), dtype=bool)
    eroded   = binary_erosion(binary_map.astype(bool), structure=struct)
    restored = binary_dilation(eroded, structure=struct) & binary_map.astype(bool)
    binary_map = restored.astype(np.uint8)
    print(f'[Stage8.5] Morph erosion r={ero_r}: {int(binary_map.sum())} px after erosion')
else:
    reason = 'sparse scene' if _sparse_scene else f'small image / r={ero_r}'
    print(f'[Stage8.5] Morph erosion skipped ({reason})')

# v22: CC min_size — scale with BOTH expected cluster area AND sparsity.
# For very sparse scenes (est_rate<0.5%), min_cc=1 so we don't kill small real clusters.
# For denser scenes, use sqrt(expected_area) × 0.25 as before.
_exp_anom_px = max(est_rate * n_pixels, 1.0)
if _sparse_scene:
    min_cc = 1   # do NOT filter by CC size for ultra-sparse scenes
else:
    min_cc = max(2, int(np.sqrt(_exp_anom_px) * 0.25))
lbl,_=sp_label(binary_map); css=np.bincount(lbl.reshape(-1))[1:] if lbl.max()>0 else np.array([])
if len(css)>0: kcc=np.where(css>=min_cc)[0]+1; bm_cc=np.isin(lbl,kcc).astype(np.uint8)
else: bm_cc=binary_map.copy()
print(f'[Stage8.5] CC min_size={min_cc} (sparse={_sparse_scene}): {int(bm_cc.sum())} px kept')
binary_map=bm_cc; n_anom=int(binary_map.sum())

gt_bin=None
if gt is not None:
    gt_bin=np.isin(gt,ANOMALY_CLASSES).astype(np.uint8) if ANOMALY_CLASSES else (gt>0).astype(np.uint8)
fig,axes=plt.subplots(1,2,figsize=(10,5))
axes[0].imshow(binary_map,cmap='gray'); axes[0].set_title('Predicted'); axes[0].axis('off')
if gt_bin is not None: axes[1].imshow(gt_bin,cmap='gray'); axes[1].set_title('GT'); axes[1].axis('off')
fig.savefig(f'{SAVE_DIR}/stage8_binary_map.png',bbox_inches='tight',dpi=100); plt.close(fig)
print('[Vis] stage8_binary_map.png')


## Stage 9 — Evaluation

In [ ]:
print('► Stage 9  — Evaluation')
if gt is not None:
    gte=np.isin(gt,ANOMALY_CLASSES).astype(np.int32) if ANOMALY_CLASSES else (gt>0).astype(np.int32)
    sf2=S_ref.reshape(-1); pf=binary_map.reshape(-1); gf=gte.reshape(-1)
    for bk in ['spec','spec_z','spat','global','latent']:
        if bk in all_scores:
            fa,ta,_=roc_curve(gf,all_scores[bk].reshape(-1))
            print(f"[Stage9] Branch '{bk}' AUC={auc(fa,ta):.4f}  robust_sep={branch_seps.get(bk,0):.3f}")
    fpr,tpr,_=roc_curve(gf,sf2); auc_val=auc(fpr,tpr)
    prec,rec,f1,_=precision_recall_fscore_support(gf,pf,average='binary',zero_division=0)
    far=float((pf[gf==0]==1).sum())/(float((gf==0).sum())+1e-8)
    tp=int((pf*gf).sum()); fp=int(((pf==1)&(gf==0)).sum()); n_gt=int(gf.sum())
    print(f'\n[Stage9] -- Evaluation Results -----------------')
    print(f'          AUC         : {auc_val:.4f}')
    print(f'          Precision   : {prec:.4f}')
    print(f'          Recall      : {rec:.4f}')
    print(f'          F1          : {f1:.4f}')
    print(f'          FAR         : {far:.4f}')
    print(f'          TP={tp}  FP={fp}  GT_anom={n_gt}')
    print(f'          DetRate={n_anom/n_pixels*100:.2f}%  GT_rate={n_gt/n_pixels*100:.2f}%')
    print(f'------------------------------------------------\n')
    fig,ax=plt.subplots(figsize=(5,5))
    ax.plot(fpr,tpr,label=f'AUC={auc_val:.4f}')
    ax.plot([0,1],[0,1],'k--'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend()
    fig.savefig(f'{SAVE_DIR}/stage9_roc.png',bbox_inches='tight',dpi=100); plt.close(fig)
    print('[Vis] stage9_roc.png')
else: print('[Stage9] No ground truth — map saved.')
print(f'\nv22 complete. Outputs -> {SAVE_DIR}/\n')